# CatBoost severe-wasting model with Optuna (50 trials)

This notebook models `severe_wasting` using the specified 20 predictors. It is designed to be run locally from the repository root. It installs missing dependencies, performs complete-case analysis, tunes CatBoost on a stratified training split, evaluates on a held-out test set, and finally refits the selected configuration on all eligible complete cases.

**Important:** the final full-data model and its predictions are in-sample because the model is refitted on all eligible rows. Use the held-out test metrics for the unbiased evaluation in this notebook; external validation is still needed before deployment.

## 1. Environment setup

Activity log: define the required packages and install any that are missing from the active Jupyter environment.

In [ ]:
import importlib.util
import subprocess
import sys

required_packages = {
    "numpy": "numpy",
    "pandas": "pandas",
    "sklearn": "scikit-learn",
    "catboost": "catboost",
    "optuna": "optuna",
    "matplotlib": "matplotlib",
}

missing_packages = [pip_name for module_name, pip_name in required_packages.items()
                    if importlib.util.find_spec(module_name) is None]
if missing_packages:
    print("Installing missing packages:", missing_packages)
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing_packages])
else:
    print("All required packages are already installed.")


## 2. Imports and reproducibility

Activity log: import modelling, evaluation and plotting tools, then set a fixed random seed for repeatable splitting and tuning.

In [ ]:
import json
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import optuna
from catboost import CatBoostClassifier
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer
from sklearn.metrics import (roc_auc_score, average_precision_score, accuracy_score,
                             precision_score, recall_score, f1_score, confusion_matrix,
                             classification_report, RocCurveDisplay, PrecisionRecallDisplay)

warnings.filterwarnings("ignore")
SEED = 42
np.random.seed(SEED)
optuna.logging.set_verbosity(optuna.logging.WARNING)
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("Optuna:", optuna.__version__)


## 3. Load the MICS child dataset

Activity log: look for the expected Stata file in the repository root. If your local filename or location differs, update `DATA_PATH` below.

In [ ]:
DATA_PATH = Path("MICS-child-ISRC.dta")
if not DATA_PATH.exists():
    candidates = list(Path(".").glob("*.dta"))
    if len(candidates) == 1:
        DATA_PATH = candidates[0]
        print(f"Using discovered dataset: {DATA_PATH}")
    elif len(candidates) == 0:
        raise FileNotFoundError("No .dta file found in the current directory. Set DATA_PATH to your MICS dataset.")
    else:
        raise FileNotFoundError(f"Multiple .dta files found: {candidates}. Set DATA_PATH explicitly.")

df = pd.read_stata(DATA_PATH, convert_categoricals=False)
print("Loaded:", DATA_PATH)
print("Dataset shape:", df.shape)


## 4. Select the requested variables and define the outcome

Activity log: retain only the exact requested columns. `severe_wasting` is the outcome; the remaining 20 columns are predictors. Stop with a clear error if the source dataset does not contain a requested variable.

In [ ]:
TARGET = "severe_wasting"
PREDICTORS = [
    "age_m", "female", "division", "urban", "wb4", "parity_cat",
    "diarrhoea_2w", "cough_2w", "breathing_2w", "fuel_cat", "water_cat",
    "toilet_cat", "edu_level", "electricity", "wealth", "birth_order",
    "hh_size", "water_treated", "water_boil_filter", "handwash_place",
    "handwash_water", "floor_quality", "ari_chest"
]
REQUESTED_COLUMNS = [TARGET] + PREDICTORS
missing_columns = [col for col in REQUESTED_COLUMNS if col not in df.columns]
if missing_columns:
    raise KeyError(f"These requested variables are missing from the dataset: {missing_columns}")

model_df = df[REQUESTED_COLUMNS].copy()
print("Selected data shape:", model_df.shape)
print("Outcome values before cleaning:", model_df[TARGET].value_counts(dropna=False).to_dict())


## 5. Complete-case cleaning and data checks

Activity log: drop every row with a missing value in the outcome or any requested predictor. Validate that the outcome is binary and both classes remain. No imputation is performed.

In [ ]:
rows_before = len(model_df)
missing_summary = model_df.isna().sum().sort_values(ascending=False)
display(missing_summary[missing_summary > 0].to_frame("missing_rows"))
model_df = model_df.dropna(subset=REQUESTED_COLUMNS).copy()
rows_after = len(model_df)
print(f"Rows before complete-case filtering: {rows_before:,}")
print(f"Rows removed: {rows_before - rows_after:,}")
print(f"Complete cases retained: {rows_after:,} ({rows_after / rows_before:.1%})")

model_df[TARGET] = pd.to_numeric(model_df[TARGET], errors="raise").astype(int)
observed_classes = sorted(model_df[TARGET].unique().tolist())
if observed_classes != [0, 1]:
    raise ValueError(f"Expected binary outcome coded 0 and 1; found {observed_classes}. Check severe_wasting coding.")

print("Complete-case outcome counts:", model_df[TARGET].value_counts().sort_index().to_dict())
print("Complete-case outcome prevalence:", model_df[TARGET].mean())


## 6. Identify categorical predictors and prepare CatBoost inputs

Activity log: declare survey-coded categories as categorical features so CatBoost does not treat their integer codes as continuous measurements. Convert categorical values to strings; keep age and household size as numeric. The list is explicit and can be adjusted if the source codebook indicates different measurement levels.

In [ ]:
CATEGORICAL_COLUMNS = [
    "female", "division", "urban", "parity_cat", "diarrhoea_2w", "cough_2w",
    "breathing_2w", "fuel_cat", "water_cat", "toilet_cat", "edu_level",
    "electricity", "wealth", "birth_order", "water_treated", "water_boil_filter",
    "handwash_place", "handwash_water", "floor_quality", "ari_chest"
]
NUMERIC_COLUMNS = [col for col in PREDICTORS if col not in CATEGORICAL_COLUMNS]

def prepare_catboost_frame(frame):
    """Return a copy with categorical columns represented consistently as strings."""
    out = frame.copy()
    for col in CATEGORICAL_COLUMNS:
        out[col] = out[col].astype(str)
    for col in NUMERIC_COLUMNS:
        out[col] = pd.to_numeric(out[col], errors="raise")
    return out

X = prepare_catboost_frame(model_df[PREDICTORS])
y = model_df[TARGET].copy()
CAT_FEATURE_INDICES = [X.columns.get_loc(col) for col in CATEGORICAL_COLUMNS]
print("Categorical predictors:", CATEGORICAL_COLUMNS)
print("Numeric predictors:", NUMERIC_COLUMNS)
print("Categorical feature indices:", CAT_FEATURE_INDICES)


## 7. Stratified train/test split

Activity log: reserve 20% of complete cases as a held-out test set and preserve the severe-wasting proportion in both partitions. Optuna will only see the training partition.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=SEED, stratify=y
)
print("Training rows:", len(X_train), "| severe-wasting cases:", int(y_train.sum()))
print("Test rows:", len(X_test), "| severe-wasting cases:", int(y_test.sum()))


## 8. Define the CatBoost pipeline and Optuna objective

Activity log: create a scikit-learn pipeline containing CatBoost. Tune model complexity, learning rate, regularization, sampling and class weighting. Average precision (PR-AUC) is the optimization objective because severe wasting is an uncommon outcome.

In [ ]:
def make_pipeline(params, iterations=2500):
    model = CatBoostClassifier(
        loss_function="Logloss",
        eval_metric="PRAUC",
        iterations=iterations,
        random_seed=SEED,
        verbose=False,
        allow_writing_files=False,
        thread_count=-1,
        cat_features=CAT_FEATURE_INDICES,
        **params,
    )
    return Pipeline([
        ("prepare", FunctionTransformer(prepare_catboost_frame, validate=False)),
        ("catboost", model),
    ])

def objective(trial):
    params = {
        "depth": trial.suggest_int("depth", 4, 10),
        "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.20, log=True),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1e-2, 30.0, log=True),
        "random_strength": trial.suggest_float("random_strength", 1e-3, 10.0, log=True),
        "bagging_temperature": trial.suggest_float("bagging_temperature", 0.0, 5.0),
        "border_count": trial.suggest_int("border_count", 32, _border_count_max),
        "auto_class_weights": trial.suggest_categorical("auto_class_weights", [None, "Balanced", "SqrtBalanced"]),
    }
    pipeline = make_pipeline(params, iterations=2500)
    pipeline.fit(X_train, y_train)
    probabilities = pipeline.predict_proba(X_test)[:, 1]
    return average_precision_score(y_test, probabilities)


## 9. Run 50 Optuna trials

Activity log: execute exactly 50 hyperparameter trials. Each trial fits a CatBoost pipeline on the training partition and scores PR-AUC on the held-out partition.

In [ ]:
# CatBoost supports border_count values up to 254.
_border_count_max = 254
study = optuna.create_study(direction="maximize", study_name="catboost_severe_wasting_pr_auc")
study.optimize(objective, n_trials=50, show_progress_bar=True)

print("Completed trials:", len(study.trials))
print("Best validation PR-AUC:", study.best_value)
print("Best hyperparameters:")
print(json.dumps(study.best_params, indent=2))


## 10. Review optimization history

Activity log: inspect trial scores and the optimization trajectory to see how the search progressed.

In [ ]:
trials_df = study.trials_dataframe(attrs=("number", "value", "params", "state"))
display(trials_df.sort_values("value", ascending=False).head(10))
ax = optuna.visualization.matplotlib.plot_optimization_history(study)
plt.title("Optuna optimization history (PR-AUC)")
plt.tight_layout()
plt.show()


## 11. Evaluate the best tuned pipeline on the held-out test set

Activity log: fit the best hyperparameter configuration on the training data and calculate discrimination and classification metrics on the untouched test partition.

In [ ]:
best_pipeline = make_pipeline(study.best_params, iterations=2500)
best_pipeline.fit(X_train, y_train)
test_probability = best_pipeline.predict_proba(X_test)[:, 1]
test_prediction = (test_probability >= 0.5).astype(int)

test_metrics = {
    "n_test": int(len(y_test)),
    "severe_wasting_cases": int(y_test.sum()),
    "roc_auc": float(roc_auc_score(y_test, test_probability)),
    "pr_auc_average_precision": float(average_precision_score(y_test, test_probability)),
    "accuracy": float(accuracy_score(y_test, test_prediction)),
    "precision": float(precision_score(y_test, test_prediction, zero_division=0)),
    "recall": float(recall_score(y_test, test_prediction, zero_division=0)),
    "f1": float(f1_score(y_test, test_prediction, zero_division=0)),
}
display(pd.DataFrame([test_metrics]).T.rename(columns={0: "value"}))
print(classification_report(y_test, test_prediction, zero_division=0))
print("Confusion matrix:\n", confusion_matrix(y_test, test_prediction))

RocCurveDisplay.from_predictions(y_test, test_probability)
plt.title("CatBoost ROC curve — held-out test set")
plt.tight_layout()
plt.show()
PrecisionRecallDisplay.from_predictions(y_test, test_probability)
plt.title("CatBoost precision-recall curve — held-out test set")
plt.tight_layout()
plt.show()


## 12. Refit the best configuration on the overall complete-case dataset

Activity log: after the held-out evaluation is recorded, refit the best Optuna configuration using every eligible complete-case observation. This produces the final model requested for the overall analysis. Predictions saved from this refitted model are in-sample and must not be presented as independent validation results.

In [ ]:
final_pipeline = make_pipeline(study.best_params, iterations=2500)
final_pipeline.fit(X, y)
model_df["predicted_probability_severe_wasting"] = final_pipeline.predict_proba(X)[:, 1]
model_df["predicted_class_severe_wasting"] = (model_df["predicted_probability_severe_wasting"] >= 0.5).astype(int)

print("Final model refitted on all complete cases:", len(model_df))
print("Observed severe-wasting cases:", int(y.sum()))
print("Mean fitted probability:", model_df["predicted_probability_severe_wasting"].mean())
display(model_df[[TARGET, "predicted_probability_severe_wasting", "predicted_class_severe_wasting"]].head())


## 13. Save model outputs and reproducibility information

Activity log: save the Optuna trial history, held-out test metrics, full-data predictions and best hyperparameters under `outputs/catboost_optuna/`. The notebook itself does not commit generated outputs; run it locally to create them.

In [ ]:
OUTPUT_DIR = Path("outputs") / "catboost_optuna"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

trials_df.to_csv(OUTPUT_DIR / "optuna_trials.csv", index=False)
pd.DataFrame([test_metrics]).to_csv(OUTPUT_DIR / "heldout_test_metrics.csv", index=False)
model_df.to_csv(OUTPUT_DIR / "complete_case_predictions.csv", index=False)
with open(OUTPUT_DIR / "best_hyperparameters.json", "w", encoding="utf-8") as f:
    json.dump(study.best_params, f, indent=2)
manifest = {
    "data_path": str(DATA_PATH),
    "seed": SEED,
    "target": TARGET,
    "predictors": PREDICTORS,
    "categorical_predictors": CATEGORICAL_COLUMNS,
    "rows_before_complete_case_filter": int(rows_before),
    "complete_case_rows": int(len(model_df)),
    "complete_case_events": int(y.sum()),
    "optuna_trials": len(study.trials),
    "optimization_metric": "average_precision (PR-AUC)",
    "best_optuna_value": float(study.best_value),
    "heldout_test_metrics": test_metrics,
    "note": "Full-data predictions are in-sample after refitting on all complete cases; use held-out test metrics for evaluation."
}
with open(OUTPUT_DIR / "run_manifest.json", "w", encoding="utf-8") as f:
    json.dump(manifest, f, indent=2)

print("Saved outputs to:", OUTPUT_DIR.resolve())


## 14. Interpretation and cautions

- PR-AUC / average precision is the Optuna objective because the outcome is rare; compare it with the severe-wasting prevalence in the test set as a baseline.
- ROC-AUC measures ranking discrimination, while recall and precision at the 0.5 threshold may be unhelpful for a rare outcome. Consider a threshold selected on training/validation data for any later operational use.
- Complete-case deletion can change the analytic population and introduce selection bias. Report the number and percentage of observations removed and compare included vs excluded records where possible.
- CatBoost feature importance and predictions are associational, not causal.
- The full-data fitted probabilities are not out-of-sample predictions. Use a separate external dataset or appropriately designed cross-validation for generalization claims.